# HELIOS — Fine-tune LLM **3B** sur les 120 incidents (Colab T4)

Vrai fine-tune de POIDS (QLoRA 4-bit) du **Qwen2.5-3B-Instruct** — le meilleur cerveau + comportement 120 gravé.
Le 4 Go local ne le tient pas ; la **T4 (16 Go)** oui, largement.

**Étapes :**
1. Exécution → Modifier le type d'exécution → **T4 GPU**.
2. **Tout exécuter**.
3. Quand la cellule 3 le demande, **glisse `corpus_120_clean.jsonl`** (depuis `scripts/train-120/`).
4. À la fin : télécharge **`aiops-sre-120-3b.gguf`** (~3 Go) et déploie sur .161 (dernière cellule).

Durée : ~30-60 min sur T4.

In [ ]:
!pip -q install -U "transformers>=4.44" "peft>=0.12" "datasets>=2.20" accelerate bitsandbytes sentencepiece
import torch
ok = torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0) if ok else 'AUCUN — active le T4 !',
      '| VRAM(GB):', round(torch.cuda.get_device_properties(0).total_memory/1e9,1) if ok else 0)

In [ ]:
# glisse corpus_120_clean.jsonl (scripts/train-120/) quand le bouton apparaît
from google.colab import files
up = files.upload()
CORPUS = list(up.keys())[0]
n = sum(1 for _ in open(CORPUS))
print('exemples SFT:', n, '| fichier:', CORPUS)

In [ ]:
import torch
from datasets import load_dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          TrainingArguments, Trainer, DataCollatorForLanguageModeling)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE = 'Qwen/Qwen2.5-3B-Instruct'
tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None: tok.pad_token = tok.eos_token

ds = load_dataset('json', data_files=CORPUS, split='train')
ds = ds.map(lambda ex: {'text': tok.apply_chat_template(ex['messages'], tokenize=False)},
            remove_columns=ds.column_names)
def tk(ex):
    out = tok(ex['text'], truncation=True, max_length=512)
    out['labels'] = out['input_ids'].copy()
    return out
ds = ds.map(tk, remove_columns=['text'])

# QLoRA 4-bit : le 3B tient dans ~2 Go de VRAM, large marge sur la T4
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map='auto')
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

collator = DataCollatorForLanguageModeling(tok, mlm=False)
args = TrainingArguments(output_dir='out', num_train_epochs=3, per_device_train_batch_size=4,
                         gradient_accumulation_steps=4, learning_rate=2e-4, warmup_ratio=0.03,
                         logging_steps=10, fp16=True, optim='paged_adamw_8bit',
                         lr_scheduler_type='cosine', report_to='none', gradient_checkpointing=True)
Trainer(model=model, args=args, train_dataset=ds, data_collator=collator).train()
model.save_pretrained('aiops-sre-120-3b-lora')
print('LoRA OK -> aiops-sre-120-3b-lora')

In [ ]:
# fusion LoRA -> modèle complet -> GGUF q8 pour Ollama
import torch, gc
del model; gc.collect(); torch.cuda.empty_cache()
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
BASE = 'Qwen/Qwen2.5-3B-Instruct'
tok = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map='cpu')
merged = PeftModel.from_pretrained(base, 'aiops-sre-120-3b-lora').merge_and_unload()
merged.save_pretrained('aiops-sre-120-3b-merged', safe_serialization=True)
tok.save_pretrained('aiops-sre-120-3b-merged')
!git clone -q https://github.com/ggerganov/llama.cpp
!pip -q install -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py aiops-sre-120-3b-merged --outfile aiops-sre-120-3b.gguf --outtype q8_0
from google.colab import files
files.download('aiops-sre-120-3b.gguf')

## Déploiement sur .161 (après téléchargement du .gguf)

Le **TEMPLATE ChatML** est OBLIGATOIRE : sans lui Ollama utilise `{{ .Prompt }}` brut et le modèle divague.

```bash
scp aiops-sre-120-3b.gguf kube@192.168.148.161:/tmp/
ssh kube@192.168.148.161
cat > /tmp/MF <<'EOF'
FROM /tmp/aiops-sre-120-3b.gguf
TEMPLATE """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}<|im_start|>user
{{ .Prompt }}<|im_end|>
<|im_start|>assistant
"""
PARAMETER stop "<|im_end|>"
PARAMETER temperature 0.15
SYSTEM "Expert SRE HELIOS. Pour tout incident Kubernetes, réponds UNIQUEMENT: ACTION=<action> | STRATEGIE=<strategie> | RAISON=<raison>."
EOF
ollama create aiops-sre-120-ft3b -f /tmp/MF
# test : curl :11434/api/chat -d '{"model":"aiops-sre-120-ft3b","messages":[{"role":"user","content":"POD_OOM le pod depasse sa limite memoire"}],"stream":false}'
```

Puis pointer le service IA vers `aiops-sre-120-ft3b` et lancer le test comparatif 120-types contre `aiops-sre-120-3b` (few-shot).